In [ ]:
import os
# Set NEAFE_DATA_FILE to the original station workbook before execution.


In [ ]:
import pandas as pd
import numpy as np
import time
from tsfresh.feature_extraction import extract_features, EfficientFCParameters
from tsfresh.utilities.dataframe_functions import roll_time_series, impute
from tsfresh import select_features
from tsfresh.feature_extraction.settings import from_columns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# 读取数据
data_path = os.environ["NEAFE_DATA_FILE"]
data = pd.read_excel(data_path, sheet_name="Sheet2")
data = data.dropna().reset_index(drop=True)

datetime_col = data.columns[0]
target_col = data.columns[-1]
data[datetime_col] = pd.to_datetime(data[datetime_col])

print("原始数据:", data.shape)

In [ ]:
data = data.sort_values(datetime_col).reset_index(drop=True)

delta = data[datetime_col].diff().dt.total_seconds().median()
samples_per_hour = 3600 / delta
WINDOW_SIZE = int(2 * samples_per_hour)

print("窗口大小:", WINDOW_SIZE)

In [ ]:
n = len(data)

train_df = data.iloc[:int(0.6*n)].copy()
valid_df = data.iloc[int(0.6*n):int(0.8*n)].copy()
test_df  = data.iloc[int(0.8*n):].copy()

print(train_df.shape, valid_df.shape, test_df.shape)

In [ ]:
start = time.time()

SAMPLE_SIZE = 2000
train_sample = train_df.iloc[:SAMPLE_SIZE].copy()
train_sample["id"] = 1

train_rolled = roll_time_series(
    train_sample,
    column_id="id",
    column_sort=datetime_col,
    max_timeshift=WINDOW_SIZE - 1,
    min_timeshift=WINDOW_SIZE - 1,   # 必须相等
    rolling_direction=1
)

fc_parameters = EfficientFCParameters()

train_features_template = extract_features(
    train_rolled,
    column_id="id",
    column_sort=datetime_col,
    default_fc_parameters=fc_parameters,
    n_jobs=1
)

print("生成特征数:", train_features_template.shape)

In [ ]:
# 1. NaN处理（必须）
train_features_template = train_features_template.replace([np.inf, -np.inf], np.nan)
train_features_template = impute(train_features_template)

# 2. 排序
train_features_template = train_features_template.sort_index()

# 3. 构造y
y = train_sample[target_col].iloc[WINDOW_SIZE-1:].reset_index(drop=True)

# 4. 强制对齐
min_len = min(len(train_features_template), len(y))
train_features_template = train_features_template.iloc[:min_len]
y = y.iloc[:min_len]

y.index = train_features_template.index

print("对齐后:", train_features_template.shape, y.shape)

# 5. 特征选择
selected = select_features(train_features_template, y, ml_task='regression')

good_cols = selected.columns.tolist()

TOP_K = 5
good_cols = good_cols[:TOP_K]

print("最终特征数:", len(good_cols))

# 转换参数
kind_to_fc = from_columns(good_cols)

In [ ]:
def compute_tsfresh(df, name):
    df = df.reset_index(drop=True)
    results = []

    for i in tqdm(range(WINDOW_SIZE-1, len(df)), desc=name):
        window = df.iloc[i-WINDOW_SIZE+1:i+1].copy()
        window["id"] = 1

        try:
            feat = extract_features(
                window,
                column_id="id",
                column_sort=datetime_col,
                kind_to_fc_parameters=kind_to_fc,
                default_fc_parameters=None,
                n_jobs=0,
                disable_progressbar=True
            )

            feat = feat.reindex(columns=good_cols)
            feat.index = [i]
            results.append(feat)

        except Exception:
            empty = pd.DataFrame(np.nan, index=[i], columns=good_cols)
            results.append(empty)

    return pd.concat(results)

In [ ]:
print("Train 特征计算")
train_ft = compute_tsfresh(train_df, "Train")

print("Valid 特征计算")
valid_ft = compute_tsfresh(valid_df, "Valid")

print("Test 特征计算")
test_ft = compute_tsfresh(test_df, "Test")

In [ ]:
# 替换inf
for df in [train_ft, valid_ft, test_ft]:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

# 用训练集均值填充
mean_vals = train_ft.mean()

train_ft = train_ft.fillna(mean_vals)
valid_ft = valid_ft.fillna(mean_vals)
test_ft  = test_ft.fillna(mean_vals)

print("最终特征维度:", train_ft.shape)

In [ ]:
end = time.time()
print("总耗时:", end - start)

In [ ]:
train_x = train_df.drop(columns=[target_col]) \
    .select_dtypes(include=[np.number]) \
    .iloc[WINDOW_SIZE-1:] \
    .reset_index(drop=True)

valid_x = valid_df.drop(columns=[target_col]) \
    .select_dtypes(include=[np.number]) \
    .iloc[WINDOW_SIZE-1:] \
    .reset_index(drop=True)

test_x = test_df.drop(columns=[target_col]) \
    .select_dtypes(include=[np.number]) \
    .iloc[WINDOW_SIZE-1:] \
    .reset_index(drop=True)

print("X shape:", train_x.shape, valid_x.shape, test_x.shape)
print("TSFresh shape:", train_ft.shape, valid_ft.shape, test_ft.shape)

train_x.to_feather("train_raw_solar5.feather")
valid_x.to_feather("valid_raw_solar5.feather")
test_x.to_feather("test_raw_solar5.feather")

train_ft.reset_index(drop=True).to_feather("train_raw_TSFreshFCP_solar5.feather")
valid_ft.reset_index(drop=True).to_feather("valid_raw_TSFreshFCP_solar5.feather")
test_ft.reset_index(drop=True).to_feather("test_raw_TSFreshFCP_solar5.feather")

# y 保存方式优化（避免列名问题）
train_df[target_col].iloc[WINDOW_SIZE-1:].reset_index(drop=True).to_frame(name=target_col).to_feather("train_y_solar5.feather")
valid_df[target_col].iloc[WINDOW_SIZE-1:].reset_index(drop=True).to_frame(name=target_col).to_feather("valid_y_solar5.feather")
test_df[target_col].iloc[WINDOW_SIZE-1:].reset_index(drop=True).to_frame(name=target_col).to_feather("test_y_solar5.feather")

print("\n=== 全部保存完成！===")


In [ ]:
import pandas as pd
import numpy as np
import optuna

from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Raw
train_raw = pd.read_feather("train_raw_solar5.feather")
valid_raw = pd.read_feather("valid_raw_solar5.feather")
test_raw  = pd.read_feather("test_raw_solar5.feather")

# tsfresh
train_neafe = pd.read_feather("train_raw_TSFreshFCP_solar5.feather")
valid_neafe = pd.read_feather("valid_raw_TSFreshFCP_solar5.feather")
test_neafe  = pd.read_feather("test_raw_TSFreshFCP_solar5.feather")

# Label
train_y = pd.read_feather("train_y_solar5.feather").values.ravel()
valid_y = pd.read_feather("valid_y_solar5.feather").values.ravel()
test_y  = pd.read_feather("test_y_solar5.feather").values.ravel()

In [ ]:
def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    nrmse = rmse / (y_true.max() - y_true.min())
    r2 = r2_score(y_true, y_pred)

    return {
        "MAE": mae,
        "MSE": mse,
        "nRMSE": nrmse,
        "R2": r2
    }

In [ ]:
from xgboost import XGBRegressor

def run_xgb(train_x, valid_x, test_x, tag):

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 800),
            "max_depth": trial.suggest_int("max_depth", 3, 8),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "subsample": trial.suggest_float("subsample", 0.7, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
            "random_state": 42,
            "n_jobs": 4,
            "tree_method": "hist"  
        }

        model = XGBRegressor(**params)

        model.fit(
            train_x, train_y,
            eval_set=[(valid_x, valid_y)],
            verbose=False
        )

        pred = model.predict(valid_x)
        return np.sqrt(mean_squared_error(valid_y, pred))


    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=10)

    best_params = study.best_params
    best_params.update({
        "random_state": 42,
        "n_jobs": 4,
        "tree_method": "hist"
    })

    # 重新训练
    best_model = XGBRegressor(**best_params)

    best_model.fit(
        train_x, train_y,
        eval_set=[(valid_x, valid_y)],
        verbose=False
    )

    preds = best_model.predict(test_x)
    metrics = evaluate(test_y, preds)
    
    print("Best Params:", best_params)
    print("Test Metrics:", metrics)
    
    return metrics 

In [ ]:
xgb_raw = run_xgb(train_raw, valid_raw, test_raw, "Raw")
xgb_ts = run_xgb(train_neafe, valid_neafe, test_neafe, "ts")
print("xgb Raw:", xgb_raw)
print("xgb tsfresh:", xgb_ts)